# A first native classifier

[`06a`](06a_variational_training.ipynb) and [`06b`](06b_pso_qng_optimizers.ipynb) trained the free parameters of a circuit with `train()` and several optimizers. This notebook applies the same idea to data: a classifier that learns to tell two groups of points apart. Instead of a circuit written by hand, the model is assembled from three pieces of `polypus.qml`, and Polypus builds the circuit from them. Since Polypus knows that circuit, it can compute the exact gradient of the cost, predict with the trained model and save it to a file.

## The data

Each sample is a point with two features, two numbers between 0 and π. One class gathers around 0.8 and the other around 2.3, with some random spread drawn from a fixed seed, so every run gets the same points. The labels are `+1` and `−1`, the convention of the loss used further down.

`qml.Dataset` keeps the samples and their labels together and checks them once: every row with the same width, every value finite. `train_test_split` sets a quarter of the samples apart, shuffled with its own seed, as a test set that the model never sees while training:

In [ ]:
import numpy as np
import polypus
from polypus import qml

rng = np.random.default_rng(3)
X = np.vstack([rng.normal(0.8, 0.3, (15, 2)), rng.normal(2.3, 0.3, (15, 2))]).clip(
    0, np.pi
)
y = np.repeat([1, -1], 15)

data = qml.Dataset(X, y)
train, test = data.train_test_split(0.25, seed=7)
print(train.num_samples, "training samples,", test.num_samples, "test samples")
print(train.num_features, "features per sample")

## The three pieces of a model

The **`Encoder`** loads each sample into the qubits. `angle_encoder("ry")` turns feature `j` into an `ry` rotation of qubit `j` by that angle, so two features need two qubits. That is why the features lie between 0 and π: an angle of 0 leaves the qubit in `0`, and an angle of π flips it to `1`. The encoder has no trainable parameters, it depends only on the data:

In [ ]:
encoder = qml.Encoder(2).angle_encoder("ry")
print(encoder.num_qubits, "qubits,", encoder.num_layers, "layer")

The **`Ansatz`** holds the trainable layers, whose angles training adjusts like the `Param` angles of [`06a`](06a_variational_training.ipynb). `real_amplitudes(1)` is a standard layout: an `ry` rotation on each qubit, a `cx` between them and another `ry` on each qubit, four free angles in total:

In [ ]:
ansatz = qml.Ansatz(2).real_amplitudes(1)
print(ansatz.num_params, "trainable parameters")

The **`Readout`** turns the measurement into a prediction. `qml.Z(0)` is the observable `Z` on qubit 0: its expected value, written `⟨Z₀⟩`, goes from `+1` when qubit 0 is always measured as `0` to `−1` when it is always `1`. With `decision="sign"` the prediction is `+1` when `⟨Z₀⟩ ≥ 0` and `−1` otherwise, one of the two labels:

In [ ]:
readout = qml.Readout([qml.Z(0)], decision="sign")
print(readout.decision, "decision on", readout.num_observables, "observable")

`qml.Model` puts the three pieces together. It checks that they fit, with the same number of qubits in the encoder and the ansatz and a readout over those qubits, and counts the trainable parameters:

In [ ]:
model = qml.Model(encoder, ansatz, readout)
print(model.num_params, "trainable parameters")

## Training with `fit`

`model.fit` trains the model with the same optimizers as `train()`. Here it is Adam, which follows the gradient of the cost. On a native model Polypus computes that gradient exactly with the parameter-shift rule: two extra circuits per trainable gate, with its angle shifted up and down, instead of an approximation.

With labels, `fit` needs a **loss**, a number that says how far each prediction is from its label. `loss="hinge"` is meant for `±1` labels: it scores each sample as `1 − y·⟨Z₀⟩`, which is 0 when `⟨Z₀⟩` equals the label `y`, 1 when `⟨Z₀⟩` is 0 and 2 when it equals the opposite label. As in [`06a`](06a_variational_training.ipynb), the optimizer maximizes a fitness, here the mean loss with its sign changed, so the best possible value is 0.

`exact=True` computes the probabilities of each circuit exactly instead of sampling shots, so the fitness carries no shot noise; [`07b`](07b_readouts_and_losses.ipynb) compares it with training on shots. The `seed` fixes the starting point, so the run repeats exactly. `train` already carries its labels, so they are not passed again:

In [ ]:
result = model.fit(
    train,
    method=polypus.Adam(max_iters=60, learning_rate=0.1),
    loss="hinge",
    exact=True,
    seed=7,
    id="first_classifier",
)
print(result.best_fitness)

## `TrainResult`'s fields

`fit` returns the same `TrainResult` as `train()` in [`06a`](06a_variational_training.ipynb). `best_params` and `best_fitness` are the best point found and its fitness, `iterations_run` counts the Adam steps actually run out of the 60 allowed by `max_iters`, and `converged` says whether it stopped early because the fitness had stopped improving. `fitness_history` keeps the best fitness after each iteration, so its curve shows how training went:

In [ ]:
import matplotlib.pyplot as plt

print(len(result.best_params), "parameters:", result.best_params)
print("iterations:", result.iterations_run, "| converged:", result.converged)
print(len(result.fitness_history), "values in fitness_history")

plt.figure(figsize=(5, 3))
plt.plot(range(1, len(result.fitness_history) + 1), result.fitness_history)
plt.xlabel("iteration")
plt.ylabel("best fitness")
plt.show()

## Predicting with `bind` and `predict`

`best_params` are just numbers. `model.bind` attaches them to the model and returns a `TrainedModel`, ready for inference, while the `Model` itself does not change. `predict` returns one decision per row, here `+1` or `−1`, and takes rows or a whole `Dataset`. Like `fit`, it accepts `exact=True`:

In [ ]:
trained = model.bind(result.best_params)
predictions = trained.predict(test, exact=True)
print("predictions:", predictions)
print("labels:     ", [int(label) for label in test.labels])

## Accuracy on the test set

The accuracy is the fraction of test samples whose prediction matches their label. These samples played no part in training, so the accuracy measures how well the model handles points it has not seen:

In [ ]:
hits = sum(p == label for p, label in zip(predictions, test.labels))
print("test accuracy:", hits / test.num_samples)

## Saving and loading

`save` writes the model, its layers and its trained parameters to a JSON file, and `qml.load` reads it back as a `TrainedModel`. Here the file goes to a temporary folder that disappears at the end of the cell. The parameters read back exactly, so the loaded model predicts the same as the original:

In [ ]:
import os
import tempfile

with tempfile.TemporaryDirectory() as folder:
    path = os.path.join(folder, "first_classifier.json")
    trained.save(path)
    loaded = qml.load(path)

print("same parameters:", loaded.params == trained.params)
print("same predictions:", loaded.predict(test, exact=True) == predictions)

## Summary

This notebook built a first native classifier: a `Dataset` split into training and test samples, a model made of an `Encoder`, an `Ansatz` and a `Readout`, trained with `fit` and Adam on the exact gradient of the hinge loss, a `TrainResult` with its `fitness_history`, predictions with `bind` and `predict`, the accuracy on samples the model had not seen, and a model saved to a file and loaded back.

The next notebook changes the readout and the loss to solve other kinds of problems.

## Next step

Continue with: [`07b_readouts_and_losses.ipynb`](07b_readouts_and_losses.ipynb).